# 01 · Preparación de los datos de BCS (fotos y bitácora)

Fase 3 de CRISP-DM. Parte de los datos que describimos en la fase 2 (fotos de las visitas del 29/09/2026 y del 14/10/2026, bitácora de expertos del 14/10 y hoja manuscrita de la primera visita) y genera el dataset listo para entrenar, con las fotos en JPG separadas por vista, un índice con el BCS de cada foto y la separación en 5 pliegues por vaca.

Las tareas siguen el orden de CRISP-DM (3.1 a 3.5), salvo la conversión a JPG, que se hace antes de construir las vistas porque la herramienta para marcarlas no puede mostrar fotos HEIC.

Esta versión corre en Google Colab. Baja los datos crudos del bucket de S3 a `/content/data`, genera ahí el dataset y al final lo compara con el que está en el bucket. Solo sube archivos al bucket si se cambia el parámetro `SUBIR` (ver la sección de carga). Los archivos originales no se modifican.

## Configuración

- Los datos no están en el repositorio sino en el bucket de S3 del reto. Primero definimos los parámetros (bucket y llaves) y después descargamos los datos crudos a `/content/data`, con la misma estructura que usa la versión local (`fotosextraidas/`, `etiquetas_bsc.xlsx`, `bitacora_hoja_manuscrita.csv` e `intermedios/vistas.csv`). Las llaves de AWS y el nombre del bucket se leen de los secretos de Colab `AWS_KEY_ID`, `AWS_SECRET` y `S3_BUCKET`, para que no queden escritos en el notebook.
- `bitacora_hoja_manuscrita.csv` es la transcripción de la hoja de BCS de la primera visita, y de ella solo se usan `id_vaca` y `bcs`.
- En Colab no se puede usar `marcar_vistas.html`, así que las vistas marcadas se bajan del bucket (`vistas.csv`). Si falta marcar alguna foto, la pausa de vistas se detiene (ver más abajo).
- Los resultados de trabajo (`fotos.csv`, `bitacora.csv` y `fotos_jpg/`) van en `/content/data/intermedios/` y el dataset final en `/content/data/dataset_bcs/`. Lo que está en `/content` se borra al cerrar la sesión de Colab, así que si necesitas algo descárgalo antes.
- `PREFIJO_DATASET` es la carpeta del bucket donde está el dataset, y `SUBIR` decide si la sección de carga sube archivos (`True`) o solo compara el dataset con el del bucket (`False`, el valor por defecto).
- Los parámetros que decidimos como equipo son `EXCLUIR_IDS` (vacas que se excluyen completas, en este caso la 2123, ver 3.2), `EXCLUIR_FOTOS` (fotos que se excluyen por su nombre, en este caso las de `1538 y 8744_atras`, ver 3.2), `K = 5` (número de pliegues) y `CALIDAD = 95` (calidad de los JPG).
- Si `dataset_bcs/` ya existe en `/content/data` (por ejemplo, de una corrida anterior en la misma sesión), el notebook guarda su índice al inicio para compararlo con el resultado al final.


In [1]:
# Parámetros
from google.colab import userdata

BUCKET = userdata.get("S3_BUCKET")
PREFIJO_FOTOS = "data/fotosextraidas/"
KEY_BITACORA = "data/csv/etiquetas_bsc.xlsx"
KEY_HOJA = "data/csv/bitacora_hoja_manuscrita.csv"
KEY_VISTAS = "bcs/intermedios/vistas.csv"
DESTINO = "/content/data"
PREFIJO_DATASET = "bcs/"
SUBIR = False

In [2]:
!pip install -q boto3 pillow-heif
import boto3
from pathlib import Path
from google.colab import userdata

s3 = boto3.client("s3", region_name="mx-central-1",
                  aws_access_key_id=userdata.get("AWS_KEY_ID"),
                  aws_secret_access_key=userdata.get("AWS_SECRET"))


def listar(prefijo):
    # Todas las llaves de archivos del bucket bajo un prefijo
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo):
        for o in pagina.get("Contents", []):
            if not o["Key"].endswith("/"):
                yield o["Key"]


def bajar(key, destino, reemplazar=False):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if reemplazar or not destino.exists():
        s3.download_file(BUCKET, key, str(destino))


data = Path(DESTINO)
for key in listar(PREFIJO_FOTOS):
    bajar(key, data / "fotosextraidas" / key[len(PREFIJO_FOTOS):])
bajar(KEY_BITACORA, data / "etiquetas_bsc.xlsx")
bajar(KEY_HOJA, data / "bitacora_hoja_manuscrita.csv")
# vistas.csv siempre se vuelve a bajar, por si se subió una versión con más marcas
bajar(KEY_VISTAS, data / "intermedios" / "vistas.csv", reemplazar=True)
print(f"Datos listos en {data}")

Datos listos en /content/data


In [3]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import datetime as dt
import hashlib
import os
import re
import shutil

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from PIL import Image, ImageOps

try:
    import pillow_heif
    pillow_heif.register_heif_opener()
    LEE_HEIC = True
except ImportError:
    LEE_HEIC = False


# Carpeta donde se descargaron los datos de S3
DATA = Path(DESTINO)
DIR_FOTOS = DATA / "fotosextraidas"
BITACORA = DATA / "etiquetas_bsc.xlsx"
HOJA = DATA / "bitacora_hoja_manuscrita.csv"
INTER = DATA / "intermedios"
DIR_JPG = INTER / "fotos_jpg"
VISTAS = INTER / "vistas.csv"
SALIDA = DATA / "dataset_bcs"

EXCLUIR_IDS = {"2123"}
# (vaca, nombre de la foto sin "(1)"): la foto muestra dos vacas y no se sabe a cuál corresponde
EXCLUIR_FOTOS = {("1538", "1538 y 8744_atras")}
K = 5
CALIDAD = 95
HILOS = 4

INTER.mkdir(exist_ok=True)
pd.set_option("display.max_rows", 120)

ref_indice = SALIDA / "indice_imagenes.csv"
referencia = pd.read_csv(ref_indice, dtype=str, keep_default_na=False) if ref_indice.is_file() else None

print("Datos en:", DATA)
for p in (DIR_FOTOS, BITACORA, HOJA, VISTAS):
    print("OK       " if p.exists() else "NO EXISTE", p.name)
print("Lectura de HEIC:", "sí" if LEE_HEIC else "no (instala pillow-heif)")
print("Índice de referencia:", f"sí ({len(referencia)} filas)" if referencia is not None else "no")

Datos en: /content/data
OK        fotosextraidas
OK        etiquetas_bsc.xlsx
OK        bitacora_hoja_manuscrita.csv
OK        vistas.csv
Lectura de HEIC: sí
Índice de referencia: no


### Funciones auxiliares

- `sha256()` calcula la huella del contenido de un archivo, porque los duplicados se detectan por contenido y no por nombre (ver 3.2).
- `es_copia_numerada()` y `nombre_base()` detectan nombres como `IMG_0530(1).HEIC` y les quitan el "(1)".
- `norm()`, `texto()` y `leer_bitacora()` leen la bitácora de Excel buscando la fila de encabezados, igual que en la fase 2.
- `limpiar_id()` y `a_numero()` pasan el ID a texto sin ".0" y el BCS a número, aceptando coma decimal.
- `leer()` lee un CSV con todo como texto y sin convertir los vacíos en `NaN`, para no alterar ningún valor.
- `clave()` ordena los IDs como números.

In [4]:
def sha256(ruta, bloque=1 << 20):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(bloque), b""):
            h.update(b)
    return h.hexdigest()


def es_copia_numerada(nombre):
    return bool(re.search(r"\(\d+\)$", Path(nombre).stem.strip()))


def nombre_base(nombre):
    return re.sub(r"\s*\(\d+\)$", "", Path(nombre).stem.strip())


def norm(t):
    t = str(t).lower().translate(str.maketrans("áéíóúüñ", "aeiouun"))
    return re.sub(r"[^a-z0-9]", "", t)


def texto(v):
    if v is None:
        return ""
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    if isinstance(v, (dt.time, dt.datetime)):
        return v.strftime("%H %M")
    return str(v).strip()


def leer_bitacora(ruta):
    wb = load_workbook(ruta, read_only=True, data_only=True)
    for ws in wb.worksheets:
        for i, fila in enumerate(ws.iter_rows(min_row=1, max_row=40, values_only=True), start=1):
            n = [norm(v) for v in fila if v not in (None, "")]
            if any(x.startswith("id") for x in n) and any(x.startswith("bcs") for x in n):
                filas = list(ws.iter_rows(min_row=i, values_only=True))
                wb.close()
                enc = [texto(c) or f"(columna {j + 1})" for j, c in enumerate(filas[0])]
                datos = []
                for num, f in enumerate(filas[1:], start=i + 1):
                    vals = [texto(c) for c in f][:len(enc)]
                    if any(vals):
                        datos.append([num] + vals + [""] * (len(enc) - len(vals)))
                return pd.DataFrame(datos, columns=["fila_excel"] + enc)
    wb.close()
    raise ValueError("No encontré la fila de encabezados con 'ID' y 'BCS'")


def columna(df, *prefijos):
    for c in df.columns:
        if c != "fila_excel" and any(norm(c).startswith(p) for p in prefijos):
            return c
    raise KeyError(prefijos)


def leer(ruta):
    df = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    df.columns = [str(c).replace("﻿", "").strip() for c in df.columns]
    return df


def limpiar_id(s):
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)


def a_numero(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ".").str.strip(), errors="coerce")


def clave(x):
    return (0, int(x)) if str(x).isdigit() else (1, str(x))

## 3.1 Seleccionar los datos

La tabla muestra qué datos entran a la preparación y por qué.

| fuente | ¿entra? | razón |
|---|---|---|
| Fotos (`fotosextraidas/`) de las visitas del 29/09 y del 14/10 | sí | son la entrada del modelo de visión |
| ID, hora y BCS de la bitácora (`etiquetas_bsc.xlsx`) | sí | el BCS de los expertos es la etiqueta |
| ID y BCS de la hoja manuscrita de la primera visita (`bitacora_hoja_manuscrita.csv`) | sí | es el BCS de expertos de las vacas fotografiadas el 29/09 |
| Columnas "Foto" y "Carpeta" de la bitácora | no | son ligas de Drive y la relación foto–vaca ya está en las carpetas |
| `vms-3.csv` | no | solo cubre parte de las vacas con BCS y no es entrada del modelo |
| "Evento BCS" de `vms-3.csv` | no | no coincide con el BCS de los expertos y todavía no sabemos qué mide (fase 2) |
| CSV de ordeño | no | son contexto del establo, no entrada del modelo |

En esta celda se hace el inventario de las fotos (con su huella) y se leen las columnas seleccionadas de la bitácora y de la hoja.

In [5]:
EXT_FOTO = {".heic", ".heif", ".jpg", ".jpeg", ".png"}
filas = []
for p in sorted(DIR_FOTOS.rglob("*")):
    if p.is_file() and p.suffix.lower() in EXT_FOTO and not p.name.startswith("."):
        rel = p.relative_to(DIR_FOTOS)
        filas.append({"id_vaca": p.parent.name.strip(), "ruta": str(rel),
                      "descarga": rel.parts[0] if len(rel.parts) > 1 else "",
                      "nombre": p.name, "ext": p.suffix.lower(), "bytes": p.stat().st_size,
                      "sha256": sha256(p), "copia_numerada": es_copia_numerada(p.name)})
todas = pd.DataFrame(filas)

bit_raw = leer_bitacora(BITACORA)
c_id, c_hora, c_bcs = columna(bit_raw, "idvaca", "id"), columna(bit_raw, "hora"), columna(bit_raw, "bcs")
hoja_raw = leer(HOJA)
print(f"Fotos leídas: {len(todas)} | Filas de la bitácora: {len(bit_raw)} | Renglones de la hoja: {len(hoja_raw)}")
print(f"Columnas seleccionadas de la bitácora: '{c_id}', '{c_hora}', '{c_bcs}'")

Fotos leídas: 761 | Filas de la bitácora: 83 | Renglones de la hoja: 27
Columnas seleccionadas de la bitácora: 'ID - VACA', 'Hora / minuto', 'BCS'


## 3.2 Limpiar los datos

### Copias exactas

Varias carpetas se descargaron dos veces de Drive, y algunas fotos venían repetidas como `IMG_0530(1).HEIC`. Dentro de cada vaca dejamos una sola copia por huella y, cuando hay varias, nos quedamos con la que no tiene "(1)" en el nombre. La columna `copias` dice cuántos archivos tenía cada foto.

In [6]:
todas = todas.sort_values(["id_vaca", "sha256", "copia_numerada", "descarga", "nombre"])
copias = todas.groupby(["id_vaca", "sha256"]).size().rename("copias")
unicas = (todas.drop_duplicates(["id_vaca", "sha256"], keep="first")
               .merge(copias, on=["id_vaca", "sha256"]).reset_index(drop=True))
print(f"Archivos: {len(todas)} | copias quitadas: {len(todas) - len(unicas)} | fotos únicas: {len(unicas)}")

Archivos: 761 | copias quitadas: 136 | fotos únicas: 625


### Exclusiones

- La vaca 2123, que está en `EXCLUIR_IDS`, tenía dos carpetas con fotos distintas que se enciman, y no se puede saber qué fotos van con cada una de sus dos evaluaciones. Como adivinar podría asignar un BCS equivocado, se excluye completa.
- Las fotos idénticas que aparecen en más de una vaca (la 1548 y la 2107) se quitan de las dos, porque no se sabe a cuál pertenecen. La 1548 conserva sus fotos propias y la 2107 se queda sin fotos.
- Las vacas 1549 y 1572 tienen fotos pero no aparecen en la bitácora ni en la hoja. Sin etiqueta no sirven para entrenar, así que sus fotos no se convierten ni se marcan. Esta regla es automática y aplica a cualquier vaca con fotos y sin BCS.
- Las fotos de `EXCLUIR_FOTOS` (`1538 y 8744_atras`) se excluyen porque muestran dos vacas y no se sabe a cuál corresponden. La 1538 conserva sus demás fotos.

Las fotos excluidas no se borran. Quedan en `fotos.csv` marcadas con `excluida = si` y con su motivo, y no se convierten a JPG.

In [7]:
# Vacas con al menos un BCS numérico en la bitácora o en la hoja
ids_con_bcs = (set(limpiar_id(bit_raw[c_id])[a_numero(bit_raw[c_bcs]).notna()])
               | set(limpiar_id(hoja_raw["id_vaca"])[a_numero(hoja_raw["bcs"]).notna()]))

vacas_por_foto = unicas.groupby("sha256")["id_vaca"].apply(lambda s: sorted(set(s)))
compartidas = vacas_por_foto[vacas_por_foto.map(len) > 1]


def motivos(fila):
    m = []
    if fila["id_vaca"] in EXCLUIR_IDS:
        m.append("vaca excluida por decision del equipo")
    if fila["id_vaca"] not in ids_con_bcs:
        m.append("vaca sin BCS")
    if (fila["id_vaca"], nombre_base(fila["nombre"])) in EXCLUIR_FOTOS:
        m.append("la foto muestra dos vacas")
    if fila["sha256"] in compartidas.index:
        m.append("misma foto en varias vacas: " + ", ".join(compartidas[fila["sha256"]]))
    return "; ".join(m)


unicas["motivo"] = unicas.apply(motivos, axis=1)
unicas["excluida"] = np.where(unicas["motivo"] != "", "si", "no")
display(unicas.loc[unicas["excluida"] == "si", "motivo"].value_counts().to_frame("fotos"))
print(f"Fotos que siguen: {(unicas['excluida'] == 'no').sum()} de {unicas.loc[unicas['excluida'] == 'no', 'id_vaca'].nunique()} vacas")

,fotos
motivo,
vaca sin BCS,20
vaca excluida por decision del equipo,14
"misma foto en varias vacas: 1548, 2107",8
la foto muestra dos vacas,3


Fotos que siguen: 580 de 77 vacas


### Bitácora

- La hora "13 00" pasa a "13:00" y las celdas con "x" quedan vacías. El valor original se conserva en `hora_original`.
- El BCS se convierte a número (acepta coma decimal) y se revisa que esté entre 1 y 5, en pasos de 0.25.
- De la hoja manuscrita se toman solo el ID y el BCS de cada renglón. Los renglones se agregan después de las filas del Excel, con `fila_excel` y la hora vacías, así que en cada vaca quedan primero las evaluaciones de la bitácora y luego las de la hoja.
- Las evaluaciones de las vacas de `EXCLUIR_IDS` se marcan con `excluida = si`.

Todo se guarda en `data/intermedios/bitacora.csv`.

In [8]:
bit = pd.DataFrame({"fila_excel": bit_raw["fila_excel"]})
bit["id_vaca"] = limpiar_id(bit_raw[c_id])
h = bit_raw[c_hora].astype(str).str.strip()
m = h.str.extract(r"^(\d{1,2})\s+(\d{2})$")
bit["hora"] = (m[0].str.zfill(2) + ":" + m[1]).where(m[0].notna(), "")
bit["hora_original"] = h
bit["bcs"] = a_numero(bit_raw[c_bcs])

# Hoja manuscrita: solo ID y BCS, después de las filas del Excel
hoja = pd.DataFrame({"fila_excel": "", "id_vaca": limpiar_id(hoja_raw["id_vaca"]), "hora": "",
                     "hora_original": "", "bcs": a_numero(hoja_raw["bcs"])})
bit = pd.concat([bit, hoja], ignore_index=True)
bit["excluida"] = np.where(bit["id_vaca"].isin(EXCLUIR_IDS), "si", "no")
bit["motivo"] = np.where(bit["excluida"] == "si", "vaca excluida por decision del equipo", "")

b = bit["bcs"]
print(f"BCS no numérico: {b.isna().sum()} | fuera de 1 a 5: {((b < 1) | (b > 5)).sum()} | "
      f"no múltiplo de 0.25: {(b.notna() & ((b * 4) % 1 != 0)).sum()}")
print(f"Evaluaciones: {len(bit)} ({len(bit) - len(hoja)} de la bitácora y {len(hoja)} de la hoja) | "
      f"excluidas: {(bit['excluida'] == 'si').sum()}")
bit.to_csv(INTER / "bitacora.csv", index=False)

BCS no numérico: 0 | fuera de 1 a 5: 0 | no múltiplo de 0.25: 0
Evaluaciones: 110 (83 de la bitácora y 27 de la hoja) | excluidas: 2


## 3.5 Formatear los datos (primera parte, conversión a JPG)

Este paso de formato va antes de 3.3 porque la vista de cada foto se marca a mano con `marcar_vistas.html`, y el navegador no puede mostrar fotos HEIC.

Tomamos estas decisiones.

- Todas las fotos pasan a JPG a resolución completa con calidad 95. JPG lo leen TensorFlow y cualquier otra herramienta, y la resolución se reduce después, al entrenar.
- Los JPG con etiqueta de orientación EXIF distinta de 1 se enderezan para que todas las librerías los vean igual. En los HEIC la librería ya aplica la rotación al abrirlos, así que no se vuelven a girar.
- Los JPG que ya venían derechos se copian sin cambios, para no perder calidad.
- Cada JPG conserva el nombre original sin el "(1)". Si dos fotos distintas de la misma vaca quedan con el mismo nombre, a la segunda se le agregan 8 caracteres de su huella.
- Cada JPG se escribe primero como `.tmp` y luego se renombra, para que no quede un archivo a medias si el proceso se corta. Si el JPG ya existe, no se vuelve a convertir.
- Se procesan 4 fotos a la vez (`HILOS`).

La conversión no guarda el mapa de profundidad de los HEIC y solo registra si la foto lo tenía.

Se guarda `data/intermedios/fotos.csv`, con una fila por foto única, incluidas las excluidas.

In [9]:
def procesar_foto(origen, destino, convertir):
    origen, ext = Path(origen), Path(origen).suffix.lower()
    r = {"ancho": None, "alto": None, "fecha_hora": "", "camara": "", "profundidad": "", "estado": "", "error": ""}
    try:
        with Image.open(origen) as im:
            exif = im.getexif()
            r["camara"] = f"{exif.get(0x010F, '')} {exif.get(0x0110, '')}".strip()
            fecha = str(exif.get_ifd(0x8769).get(0x9003, "") or exif.get(0x0132, "")).strip()
            r["fecha_hora"] = fecha.replace(":", "-", 2) if re.match(r"^\d{4}:\d{2}:\d{2}", fecha) else fecha
            if ext in {".heic", ".heif"}:
                r["profundidad"] = "si" if im.info.get("depth_images") else "no"
                girar = False
            else:
                r["profundidad"] = "no"
                girar = exif.get(0x0112, 1) not in (None, 1)
            final = ImageOps.exif_transpose(im) if girar else im
            r["ancho"], r["alto"] = final.size
            if not convertir:
                return {**r, "estado": "excluida"}
            destino = Path(destino)
            if destino.exists():
                return {**r, "estado": "ya existia"}
            destino.parent.mkdir(parents=True, exist_ok=True)
            tmp = destino.with_suffix(".tmp")
            if ext in {".jpg", ".jpeg"} and not girar:
                shutil.copy2(origen, tmp)
                r["estado"] = "copiada"
            else:
                exif[0x0112] = 1
                extra = {"quality": CALIDAD, "exif": exif.tobytes()}
                if im.info.get("icc_profile"):
                    extra["icc_profile"] = im.info["icc_profile"]
                final.convert("RGB").save(tmp, "JPEG", **extra)
                r["estado"] = "convertida"
            os.replace(tmp, destino)
    except Exception as e:
        r["estado"], r["error"] = "error", str(e).replace(str(origen), origen.name)[:200]
    return r


# Nombre de destino: original sin "(1)"; si choca, se agregan 8 caracteres de la huella
destinos, usados = [], set()
for _, f in unicas.iterrows():
    base = nombre_base(f["nombre"])
    if (f["id_vaca"], base.lower()) in usados:
        base = f"{base}_{f['sha256'][:8]}"
    usados.add((f["id_vaca"], base.lower()))
    destinos.append(f"fotos_jpg/{f['id_vaca']}/{base}.jpg")
unicas["archivo_jpg"] = np.where(unicas["excluida"] == "no", destinos, "")

tareas = [(DIR_FOTOS / f["ruta"], INTER / f["archivo_jpg"] if f["archivo_jpg"] else "", f["excluida"] == "no")
          for _, f in unicas.iterrows()]
with ThreadPoolExecutor(max_workers=HILOS) as ex:
    meta = pd.DataFrame(list(ex.map(lambda t: procesar_foto(*t), tareas)))
for c in ("ancho", "alto"):
    meta[c] = pd.to_numeric(meta[c], errors="coerce").astype("Int64")
fotos = pd.concat([unicas, meta], axis=1)
fotos.loc[fotos["estado"] == "error", "archivo_jpg"] = ""

fotos = (fotos.rename(columns={"ruta": "archivo_original", "ext": "formato_original", "bytes": "bytes_original"})
         [["id_vaca", "archivo_jpg", "excluida", "motivo", "archivo_original", "copias", "formato_original",
           "bytes_original", "sha256", "fecha_hora", "camara", "ancho", "alto", "profundidad", "estado", "error"]]
         .sort_values(["id_vaca", "archivo_original"]).reset_index(drop=True))
fotos.to_csv(INTER / "fotos.csv", index=False)

display(fotos["estado"].value_counts().to_frame("fotos"))
listas = fotos[(fotos["excluida"] == "no") & (fotos["archivo_jpg"] != "")]
tam = sum((INTER / r).stat().st_size for r in listas["archivo_jpg"] if (INTER / r).exists())
print(f"JPG listos: {len(listas)} ({tam / 1e9:.2f} GB) | con error: {(fotos['estado'] == 'error').sum()}")

,fotos
estado,
convertida,475
copiada,105
excluida,45


JPG listos: 580 (2.80 GB) | con error: 0


## Pausa para marcar la vista de cada foto

La vista de cada foto no viene en los nombres ni en los metadatos, así que se marca a mano con la página `src/bigdata/herramientas/marcar_vistas.html`. Esa página corre en el navegador con las fotos de tu computadora y por eso en Colab no se marca. Aquí el notebook usa el `vistas.csv` que se bajó del bucket (`KEY_VISTAS`).

Si falta marcar alguna foto, esta celda se detiene y dice cuántas faltan por vaca. En ese caso hay que seguir estos pasos.

1. Corre la versión local del notebook hasta esta pausa y marca las fotos que faltan con `marcar_vistas.html`. Carga primero el `vistas.csv` con el botón "Cargar vistas.csv previo" y usa la tecla N para ir a la siguiente foto sin marcar.
2. Sube el `vistas.csv` nuevo al bucket, en la llave de `KEY_VISTAS`.
3. Vuelve a correr la celda de descarga y el notebook desde aquí.

Si `vistas.csv` tiene todas las fotos marcadas, esta celda solo lo confirma y el notebook sigue.

In [10]:
assert VISTAS.is_file(), (
    f"Falta {VISTAS}. Revisa que la llave {KEY_VISTAS} exista en el bucket y vuelve a correr la celda de descarga.")

# Todas las fotos que se usan deben estar marcadas (una vista o descartada)
por_marcar = leer(INTER / "fotos.csv")
por_marcar = por_marcar[(por_marcar["excluida"] == "no") & (por_marcar["archivo_jpg"] != "")]
vis_p = leer(VISTAS)
marcadas = set(vis_p.loc[vis_p["usable"].isin(["si", "no"]), "archivo_jpg"])
faltan = por_marcar[~por_marcar["archivo_jpg"].isin(marcadas)]
if len(faltan):
    conteo = faltan.groupby("id_vaca").size()
    detalle = ", ".join(f"{v} ({conteo[v]})" for v in sorted(conteo.index, key=clave))
    raise RuntimeError(
        f"Faltan {len(faltan)} fotos por marcar en vistas.csv. Por vaca: {detalle}. "
        f"Márcalas con marcar_vistas.html en la versión local, sube el vistas.csv a {KEY_VISTAS}, "
        "vuelve a correr la celda de descarga y el notebook desde aquí.")
print(f"vistas.csv encontrado: {VISTAS} | {len(por_marcar)} fotos, todas marcadas")

vistas.csv encontrado: /content/data/intermedios/vistas.csv | 580 fotos, todas marcadas


## 3.3 Construir datos nuevos

### Vista y si la foto es usable

Se unen las fotos no excluidas con `vistas.csv` usando `archivo_jpg`, y cada foto queda en una carpeta según su marca.

| marca | `usable` | carpeta |
|---|---|---|
| posterior, lateral izquierda o lateral derecha | si | la vista (`posterior`, `lateral_izq`, `lateral_der`) |
| descartada | no | `descartadas` |
| sin marca | (vacío) | `sin_marcar` (no se usa y, con la revisión de la pausa, no debería haber ninguna) |

Las descartadas no se borran. Se guardan aparte para que se puedan revisar, pero no se entrenan.

In [11]:
fotos_i = leer(INTER / "fotos.csv")
bit_i = leer(INTER / "bitacora.csv")
vis = leer(VISTAS)

f = fotos_i[(fotos_i["excluida"] == "no") & (fotos_i["archivo_jpg"] != "")].copy()
fv = f.merge(vis[["archivo_jpg", "vista", "usable", "marcado_por", "marcado_en"]], on="archivo_jpg", how="left")
for c in ("vista", "usable", "marcado_por", "marcado_en"):
    fv[c] = fv[c].fillna("")
fv["carpeta"] = np.where(fv["usable"] == "si", fv["vista"],
                         np.where(fv["usable"] == "no", "descartadas", "sin_marcar"))
fv["ruta"] = "imagenes/" + fv["carpeta"] + "/" + fv["id_vaca"] + "/" + fv["archivo_jpg"].str.split("/").str[-1]

display(fv["carpeta"].value_counts().to_frame("fotos"))
print(f"Marcas en vistas.csv que no corresponden a ninguna foto: {(~vis['archivo_jpg'].isin(f['archivo_jpg'])).sum()}")

,fotos
carpeta,
posterior,257
lateral_der,148
lateral_izq,146
descartadas,29


Marcas en vistas.csv que no corresponden a ninguna foto: 0


### Etiqueta de BCS

A algunas vacas las evaluaron dos o tres veces, ya sea expertos distintos el mismo día o en las dos visitas. Decidimos no elegir una evaluación sobre otra y armamos estas columnas.

- `bcs_1`, `bcs_2` y `bcs_3` guardan las evaluaciones en orden, primero las de la bitácora y después las de la hoja. El orden no indica qué experto dio cada una, y quedan vacías si la vaca tiene menos evaluaciones.
- `n_eval` es el número de evaluaciones de la vaca.
- `bcs_entrenamiento` es el promedio de todas las evaluaciones de la vaca, de las dos visitas, y es la etiqueta con la que se entrena. Si solo hay una evaluación, es esa. Todas las fotos de la vaca usan este valor, sin importar la fecha de la foto.
- `dif_evaluadores` es la diferencia entre la evaluación mayor y la menor, cuando hay dos o más.

Para evaluar el modelo se compara contra cada evaluación real (`evaluaciones.csv`) y no contra el promedio.

También se cuentan las fotos usables por vista de cada vaca. Una vaca entra al entrenamiento solo si tiene al menos una foto usable y BCS.

In [12]:
b = bit_i[bit_i["excluida"] == "no"].copy()
b["bcs_num"] = pd.to_numeric(b["bcs"], errors="coerce")
b = b[b["bcs_num"].notna()].copy()
# Orden de las evaluaciones: el de bitacora.csv (primero el Excel, después la hoja)
b["_pos"] = b.index
b = b.sort_values(["id_vaca", "_pos"])
b["orden_evaluacion"] = b.groupby("id_vaca").cumcount() + 1
cb = b.groupby("id_vaca").agg(n_eval=("bcs_num", "size"), bcs_entrenamiento=("bcs_num", "mean"),
                              bcs_min=("bcs_num", "min"), bcs_max=("bcs_num", "max"))
cb["bcs_1"] = b[b["orden_evaluacion"] == 1].set_index("id_vaca")["bcs_num"]
cb["bcs_2"] = b[b["orden_evaluacion"] == 2].set_index("id_vaca")["bcs_num"]
cb["bcs_3"] = b[b["orden_evaluacion"] == 3].set_index("id_vaca")["bcs_num"]
assert cb["n_eval"].max() <= 3, "Hay vacas con más de 3 evaluaciones; agrega una columna bcs_4."
cb["dif_evaluadores"] = (cb["bcs_max"] - cb["bcs_min"]).where(cb["n_eval"] >= 2)
cb = cb.drop(columns=["bcs_min", "bcs_max"]).reset_index()

us = fv["usable"] == "si"
fv["_u"] = us.astype(int)
fv["_post"] = (us & (fv["vista"] == "posterior")).astype(int)
fv["_izq"] = (us & (fv["vista"] == "lateral_izq")).astype(int)
fv["_der"] = (us & (fv["vista"] == "lateral_der")).astype(int)
vac = fv.groupby("id_vaca").agg(n_fotos=("ruta", "size"), n_usables=("_u", "sum"), n_posterior=("_post", "sum"),
                                n_lateral_izq=("_izq", "sum"), n_lateral_der=("_der", "sum")).reset_index()
vac = vac.merge(cb, on="id_vaca", how="left")

sin_usables = vac.loc[vac["n_usables"] == 0, "id_vaca"].tolist()
dif = cb["dif_evaluadores"].dropna()
print(f"Vacas: {len(vac)} | sin fotos usables: {sin_usables}")
print(f"Vacas con dos o más evaluaciones: {len(dif)} (con tres: {(cb['n_eval'] == 3).sum()}) | "
      f"diferencia media {dif.mean():.3f}, máxima {dif.max():.2f}, idénticas {(dif == 0).sum()}")
print(f"Vacas sin ninguna foto lateral usable: {((vac['n_lateral_izq'] == 0) & (vac['n_lateral_der'] == 0) & (vac['n_usables'] > 0)).sum()}")

Vacas: 77 | sin fotos usables: ['1554']
Vacas con dos o más evaluaciones: 27 (con tres: 3) | diferencia media 0.185, máxima 0.50, idénticas 11
Vacas sin ninguna foto lateral usable: 18


## Diseño de prueba con validación cruzada k-fold por vaca

En CRISP-DM esta es la tarea 4.2 (Generar el diseño de prueba, fase de Modelado). La incluimos aquí para que el dataset salga listo para entrenar.

Tomamos estas decisiones.

- La separación es por vaca y no por foto, así que todas las fotos de una vaca quedan en el mismo pliegue. Si una vaca tuviera fotos en entrenamiento y en prueba, el modelo podría aprender a reconocerla en lugar de aprender su BCS, y las métricas saldrían infladas (data leakage).
- Los pliegues no son aleatorios. Ordenamos las vacas por `bcs_entrenamiento` (y por ID cuando empatan) y las asignamos en rotación (0, 1, 2, 3, 4, 0, 1…), de modo que cada pliegue tiene BCS bajos, medios y altos y el reparto sale igual cada vez que se corre.
- Solo reciben pliegue las vacas con al menos una foto usable y BCS.

In [13]:
eleg = vac[(vac["n_usables"] >= 1) & vac["bcs_entrenamiento"].notna()].copy()
eleg["_idn"] = pd.to_numeric(eleg["id_vaca"], errors="coerce")
eleg = eleg.sort_values(["bcs_entrenamiento", "_idn", "id_vaca"], na_position="last")
eleg["fold"] = np.arange(len(eleg)) % K
vac = vac.merge(eleg[["id_vaca", "fold"]], on="id_vaca", how="left")
vac["fold"] = vac["fold"].astype("Int64")
vac["en_entrenamiento"] = np.where(vac["fold"].notna(), "si", "no")
vac["n_eval"] = vac["n_eval"].astype("Int64")

resumen_folds = (vac[vac["en_entrenamiento"] == "si"].groupby("fold")
                 .agg(vacas=("id_vaca", "size"), fotos=("n_usables", "sum"), posterior=("n_posterior", "sum"),
                      lateral_izq=("n_lateral_izq", "sum"), lateral_der=("n_lateral_der", "sum"),
                      bcs_medio=("bcs_entrenamiento", "mean")).round({"bcs_medio": 2}))
display(resumen_folds)

,vacas,fotos,posterior,lateral_izq,lateral_der,bcs_medio
fold,,,,,,
0,16,146,67,43,36,2.99
1,15,69,42,16,11,2.96
2,15,113,49,25,39,2.98
3,15,138,61,39,38,3.02
4,15,85,38,23,24,3.02


## 3.4 Integrar los datos

Las fotos, las vistas, el BCS y los pliegues se unen en tres tablas.

- `indice_imagenes.csv` tiene una fila por foto, con su ruta, vista, BCS y pliegue. Para entrenar se usan las filas con `usar_en_entrenamiento = si`, que son las fotos usables de vacas con pliegue.
- `vacas.csv` tiene una fila por vaca, con su pliegue, su BCS y cuántas fotos tiene de cada vista.
- `evaluaciones.csv` tiene una fila por evaluación real de BCS y sirve para evaluar el modelo. Incluye las vacas que no entran al entrenamiento, marcadas con `en_entrenamiento = no`.

Las tres tablas tienen las mismas columnas que generan los scripts de `src/bigdata/pyspark/` del repositorio, incluida `bcs_3`.

In [14]:
COLS_INDICE = ["ruta", "id_vaca", "vista", "usable", "usar_en_entrenamiento", "fold", "bcs_entrenamiento",
               "bcs_1", "bcs_2", "bcs_3", "n_eval", "dif_evaluadores", "archivo_jpg", "archivo_original", "sha256",
               "fecha_hora", "camara", "ancho", "alto", "profundidad", "marcado_por", "marcado_en"]
COLS_VACAS = ["id_vaca", "en_entrenamiento", "fold", "bcs_entrenamiento", "bcs_1", "bcs_2", "bcs_3", "n_eval",
              "dif_evaluadores", "n_fotos", "n_usables", "n_posterior", "n_lateral_izq", "n_lateral_der"]
COLS_EVAL = ["fila_excel", "id_vaca", "hora", "bcs", "orden_evaluacion", "n_eval", "fold", "en_entrenamiento"]

datos_vaca = vac[["id_vaca", "fold", "en_entrenamiento", "bcs_entrenamiento", "bcs_1", "bcs_2", "bcs_3", "n_eval", "dif_evaluadores"]]
idx = fv.merge(datos_vaca, on="id_vaca", how="left")
idx["usar_en_entrenamiento"] = np.where((idx["usable"] == "si") & (idx["en_entrenamiento"] == "si"), "si", "no")

ev = b.merge(cb[["id_vaca", "n_eval"]], on="id_vaca", how="left")
ev = ev.merge(vac[["id_vaca", "fold", "en_entrenamiento"]], on="id_vaca", how="left")
ev["en_entrenamiento"] = ev["en_entrenamiento"].fillna("no")
ev["bcs"] = ev["bcs_num"]
ev["n_eval"] = ev["n_eval"].astype("Int64")


def por_id(df, extra):
    df = df.assign(_idn=pd.to_numeric(df["id_vaca"], errors="coerce"))
    return df.sort_values(["_idn", "id_vaca"] + extra, na_position="last")


indice = por_id(idx, ["ruta"])[COLS_INDICE]
vacas = por_id(vac, [])[COLS_VACAS]
evaluaciones = ev.sort_values("_pos")[COLS_EVAL]

usar = indice[indice["usar_en_entrenamiento"] == "si"]
print(f"indice_imagenes: {len(indice)} fotos | para entrenar: {len(usar)} de {usar['id_vaca'].nunique()} vacas")
print(f"vacas: {len(vacas)} | con pliegue: {(vacas['en_entrenamiento'] == 'si').sum()}")
print(f"evaluaciones: {len(evaluaciones)} | de vacas en entrenamiento: {(evaluaciones['en_entrenamiento'] == 'si').sum()}")

indice_imagenes: 580 fotos | para entrenar: 551 de 76 vacas
vacas: 77 | con pliegue: 76
evaluaciones: 108 | de vacas en entrenamiento: 106


## 3.5 Formatear los datos (segunda parte, dataset final)

Se arma `/content/data/dataset_bcs/` con esta estructura. La sección de carga lo compara con el del bucket y, si se pide, lo sube.

```
dataset_bcs/
├── imagenes/<vista>/<id_vaca>/<foto>.jpg
├── indice_imagenes.csv
├── vacas.csv
└── evaluaciones.csv
```

Tomamos estas decisiones.

- Las fotos se copian y no se mueven, para que `data/intermedios/` quede completo y se pueda volver a correr.
- Si una foto ya está en el destino con el mismo tamaño, no se copia de nuevo.
- Al final se revisa que cada ruta del índice exista. Si en el destino sobran archivos de una corrida anterior, solo se avisa y no se borra nada.
- El `LEEME.md` y el `tiempos.csv` del bucket no se generan aquí, porque el primero lo mantiene el equipo y el segundo viene de la comparación con PySpark (ver la sección de PySpark).

In [15]:
SALIDA.mkdir(exist_ok=True)
copiadas = iguales = 0
for fila in indice.itertuples(index=False):
    src, dst = INTER / fila.archivo_jpg, SALIDA / fila.ruta
    if dst.is_file() and dst.stat().st_size == src.stat().st_size:
        iguales += 1
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    copiadas += 1

indice.to_csv(SALIDA / "indice_imagenes.csv", index=False)
vacas.to_csv(SALIDA / "vacas.csv", index=False)
evaluaciones.to_csv(SALIDA / "evaluaciones.csv", index=False)

esperadas = {str(Path(r)) for r in indice["ruta"]}
en_destino = {str(p.relative_to(SALIDA)) for p in (SALIDA / "imagenes").rglob("*") if p.is_file()}
faltan, sobran = esperadas - en_destino, en_destino - esperadas
tam = sum(p.stat().st_size for p in SALIDA.rglob("*") if p.is_file())
print(f"Fotos copiadas ahora: {copiadas} | ya estaban: {iguales} | tamaño del dataset: {tam / 1e9:.2f} GB")
print(f"Rutas del índice que faltan: {len(faltan)} | archivos que sobran: {len(sobran)}")
display(indice["ruta"].str.split("/").str[1].value_counts().to_frame("fotos por carpeta"))

Fotos copiadas ahora: 580 | ya estaban: 0 | tamaño del dataset: 2.80 GB
Rutas del índice que faltan: 0 | archivos que sobran: 0


,fotos por carpeta
ruta,
posterior,257
lateral_der,148
lateral_izq,146
descartadas,29


## Verificación contra la referencia

Si al inicio ya existía un `indice_imagenes.csv` en `data/dataset_bcs/`, por ejemplo el que se generó con los scripts y está en S3, aquí se compara con el nuevo. Si coinciden, el notebook reproduce exactamente el dataset del bucket. Si el índice del bucket es de la versión anterior (sin las fotos del 29/09, sin la hoja y sin `bcs_3`), lo esperado es que no coincida, y la celda dice qué rutas se agregaron o quitaron y qué columnas cambiaron en las fotos que están en los dos. Las columnas numéricas se comparan como números, para que "3.0" y "3" cuenten igual.

In [16]:
NUMERICAS = {"fold", "bcs_entrenamiento", "bcs_1", "bcs_2", "bcs_3", "n_eval", "dif_evaluadores", "ancho", "alto"}
if referencia is None:
    print("No había índice de referencia; no hay nada que comparar.")
else:
    nuevo = leer(SALIDA / "indice_imagenes.csv")
    ref = referencia
    print(f"Filas: nuevo {len(nuevo)} | referencia {len(ref)}")
    print(f"Columnas nuevas: {[c for c in nuevo.columns if c not in ref.columns]} | "
          f"quitadas: {[c for c in ref.columns if c not in nuevo.columns]}")
    print(f"Rutas nuevas: {len(set(nuevo['ruta']) - set(ref['ruta']))} | quitadas: {len(set(ref['ruta']) - set(nuevo['ruta']))}")
    comunes = [c for c in nuevo.columns if c in ref.columns and c != "ruta"]
    m = nuevo.merge(ref, on="ruta", suffixes=("", "_ref"))
    difs = []
    for c in comunes:
        if c in NUMERICAS:
            a, r = pd.to_numeric(m[c], errors="coerce"), pd.to_numeric(m[c + "_ref"], errors="coerce")
            malas = ~(((a - r).abs() < 1e-9) | (a.isna() & r.isna()))
        else:
            malas = m[c] != m[c + "_ref"]
        if malas.any():
            difs.append(f"{c}: {int(malas.sum())} diferencias")
    iguales = not difs and len(nuevo) == len(ref) and list(nuevo.columns) == list(ref.columns)
    print("El índice coincide con la referencia." if iguales else
          f"En las {len(m)} fotos que están en los dos: " + ("; ".join(difs) if difs else "sin diferencias"))

No había índice de referencia; no hay nada que comparar.


## Carga del dataset a S3

La primera carga del dataset al bucket se hizo a mano desde la consola de S3, subiendo la carpeta `dataset_bcs/` completa. Esta sección reproduce esa carga y la verifica.

- Con `SUBIR = False`, el valor por defecto, no sube nada. Compara cada archivo de `/content/data/dataset_bcs/` con el que está en `PREFIJO_DATASET` del bucket, primero por tamaño y después por su huella MD5, que S3 guarda como ETag cuando el archivo se subió en un solo envío. Si un CSV tiene bytes distintos, se baja y se compara su contenido, para que una diferencia de formato como "3.0" contra "3" no cuente como error.
- Con `SUBIR = True` sube solo los archivos que faltan o que cambiaron, y después vuelve a comparar. Los que ya están iguales no se tocan.

Los archivos que solo están en el bucket, como `LEEME.md` y `tiempos.csv`, se reportan aparte y no se borran. Al final se guarda y se descarga `carga_resumen.json`, con conteos y sin nombres de fotos, para la carpeta `reports/bigdata/resultados/` del repositorio.

In [17]:
# Carga a S3: compara el dataset de Colab con el del bucket y, si SUBIR es True, sube lo que falte o cambió
import json
import tempfile
from boto3.s3.transfer import TransferConfig
from google.colab import files

prefijo = PREFIJO_DATASET.strip("/") + "/"
config = TransferConfig(multipart_threshold=1024 ** 3)  # un solo envío por archivo, así la ETag es el MD5


def md5(ruta, bloque=1 << 20):
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(bloque), b""):
            h.update(b)
    return h.hexdigest()


def en_bucket():
    # Ruta dentro del dataset -> (tamaño, ETag)
    return {o["Key"][len(prefijo):]: (o["Size"], o["ETag"].strip('"'))
            for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo)
            for o in pagina.get("Contents", []) if not o["Key"].endswith("/")}


def mismo_contenido(rel):
    # Compara un CSV de Colab con el del bucket como tabla, con las columnas numéricas como números
    with tempfile.TemporaryDirectory() as tmp:
        s3.download_file(BUCKET, prefijo + rel, f"{tmp}/remoto.csv")
        a, b = leer(SALIDA / rel), leer(f"{tmp}/remoto.csv")
    if list(a.columns) != list(b.columns) or len(a) != len(b):
        return False
    for c in a.columns:
        na, nb = pd.to_numeric(a[c], errors="coerce"), pd.to_numeric(b[c], errors="coerce")
        iguales = (a[c] == b[c]) | ((na - nb).abs() < 1e-9)
        if not iguales.all():
            return False
    return True


def comparar(locales, remotos):
    estado = {}
    for rel, ruta in locales.items():
        if rel not in remotos:
            estado[rel] = "falta en el bucket"
            continue
        tam, etag = remotos[rel]
        if "-" not in etag and md5(ruta) == etag:
            estado[rel] = "igual"
        elif "-" in etag and tam == ruta.stat().st_size:
            estado[rel] = "igual en tamaño"  # se subió en partes y la ETag no es el MD5
        elif rel.endswith(".csv") and mismo_contenido(rel):
            estado[rel] = "igual en contenido"
        else:
            estado[rel] = "distinto"
    return estado


locales = {p.relative_to(SALIDA).as_posix(): p for p in sorted(SALIDA.rglob("*")) if p.is_file()}
remotos = en_bucket()
estado = comparar(locales, remotos)
subidos = 0
if SUBIR:
    for rel, e in estado.items():
        if e in ("falta en el bucket", "distinto"):
            s3.upload_file(str(locales[rel]), BUCKET, prefijo + rel, Config=config)
            subidos += 1
    remotos = en_bucket()
    estado = comparar(locales, remotos)

conteo = pd.Series(estado).value_counts()
solo_bucket = sorted(set(remotos) - set(locales))
otros = [r for r in solo_bucket if not r.startswith("imagenes/")]
completa = all(e.startswith("igual") for e in estado.values())
display(conteo.to_frame("archivos"))
print(f"Modo: {'subir' if SUBIR else 'solo comparar'} | subidos ahora: {subidos} | "
      f"fotos del bucket que no están en Colab: {len(solo_bucket) - len(otros)} | otros archivos solo en el bucket: {otros}")
print("El bucket tiene el mismo dataset que generó este notebook." if completa else
      "El bucket NO tiene el mismo dataset. Revisa la tabla de arriba.")

resumen_carga = {"fecha_utc": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M"), "prefijo": prefijo,
                 "modo": "subir" if SUBIR else "solo comparar", "archivos_en_colab": len(locales),
                 "archivos_en_bucket": len(remotos), "gb_en_colab": round(sum(p.stat().st_size for p in locales.values()) / 1e9, 2),
                 "por_estado": {k: int(v) for k, v in conteo.items()}, "subidos": subidos,
                 "fotos_solo_en_bucket": len(solo_bucket) - len(otros), "otros_archivos_solo_en_bucket": otros,
                 "carga_completa": completa}
Path("/content/carga_resumen.json").write_text(json.dumps(resumen_carga, ensure_ascii=False, indent=2), encoding="utf-8")
files.download("/content/carga_resumen.json")

,archivos
igual,583


Modo: solo comparar | subidos ahora: 0 | fotos del bucket que no están en Colab: 0 | otros archivos solo en el bucket: ['intermedios/bitacora.csv', 'intermedios/fotos.csv', 'intermedios/vistas.csv', 'tiempos.csv']
El bucket tiene el mismo dataset que generó este notebook.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Versión con PySpark

El paso de integración (3.3, diseño de prueba y 3.4) también se programó con PySpark, para practicar un enfoque que serviría si el volumen creciera. Esa versión no está en este notebook porque necesita Java y Spark, y sus scripts están en la carpeta `src/bigdata/pyspark/` del repositorio.

- `reglas.py` tiene las columnas y las constantes que comparten las dos versiones.
- `indice_pandas.py` tiene la misma lógica de este notebook, como script.
- `indice_spark.py` es la versión con PySpark, y corre en modo local o en un clúster.
- `preparar_entradas.py` arma las entradas de los dos scripts a partir del dataset del bucket.
- `comparar_tiempos.py` corre las dos versiones varias veces, mide tiempo, CPU y memoria, y verifica que den exactamente las mismas tablas.

La comparación con los datos actuales se corre en `src/notebooks/bigdata/medicion_recursos.ipynb`, y sus resultados quedan en `reports/bigdata/resultados/tiempos_recursos.csv`. En la medición que se hizo con la versión anterior del dataset, PySpark tardó unas 40 veces más que pandas por el costo de arrancar Spark, y por eso este notebook usa pandas.

## Conclusión

Las cifras de cada tarea están en las salidas de este notebook. Su análisis y las decisiones que se tomaron están en el reporte de Preparación de los datos.